# Donut Inference Benchmark — Analysis

This notebook analyses results from all six benchmark experiments.
Run the corresponding shell script before executing each section.

| Section | Script | Question |
|---------|--------|----------|
| 1 | exp01_throughput_sweep.sh | What is the peak throughput and optimal batch size? |
| 2 | exp02_stability.sh | Are our measurements stable? How many warmup runs are needed? |
| 3 | exp03_phase_breakdown.sh | Where does inference time go? |
| 4 | exp04_cpu_baseline.sh | How much faster is the GPU? |
| 5 | exp05_decoder_efficiency.sh | How much throughput does document diversity cost? |
| 6 | exp06_sustained_throughput.sh | Does performance degrade over long runs? |
| 7 | (derived from exp01) | What is the fastest way to process 100 documents? |

In [ ]:
from __future__ import annotations

import glob
import math
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="husl", font_scale=1.15)
plt.rcParams.update({"figure.dpi": 130})

LOGS = Path("../logs")
FIGS = LOGS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

print(f"Logs root: {LOGS.resolve()}")

In [ ]:
# ── Data-loading helpers ──────────────────────────────────────────────────────

def _latest(pattern: str) -> Path | None:
    files = sorted(glob.glob(pattern))
    return Path(files[-1]) if files else None


def load_sweep(exp_dir: str) -> pd.DataFrame:
    """Load the most-recent sweep CSV from an experiment directory."""
    f = _latest(str(LOGS / exp_dir / "*_sweep.csv"))
    if f is None:
        raise FileNotFoundError(
            f"No sweep CSV in logs/{exp_dir}/  — run the experiment script first."
        )
    df = pd.read_csv(f)
    # Drop OOM rows and ensure numeric batch_size
    df = df[df["batch_size"].astype(str) != "OOM"].copy()
    df["batch_size"] = df["batch_size"].astype(int)
    return df.sort_values("batch_size").reset_index(drop=True)


def load_runs(exp_dir: str, bs: int | None = None) -> pd.DataFrame:
    """Concatenate all per-run CSVs from an experiment (optionally for one BS)."""
    pat = f"*_bs{bs}_runs.csv" if bs else "*_bs*_runs.csv"
    files = sorted(glob.glob(str(LOGS / exp_dir / pat)))
    if not files:
        raise FileNotFoundError(f"No run CSVs in logs/{exp_dir}/ matching {pat}")
    dfs = []
    for fp in files:
        tmp = pd.read_csv(fp)
        if "batch_size" not in tmp.columns:
            m = re.search(r"_bs(\d+)_runs", fp)
            if m:
                tmp["batch_size"] = int(m.group(1))
        dfs.append(tmp)
    return pd.concat(dfs, ignore_index=True)


def load_single_run_csv(exp_dir: str) -> pd.DataFrame:
    """Load the most-recent single-batch run CSV (no _bs* suffix)."""
    # Exclude sweep files
    files = [f for f in sorted(glob.glob(str(LOGS / exp_dir / "*.csv")))
             if "_bs" not in f and "_sweep" not in f]
    if not files:
        raise FileNotFoundError(f"No single-run CSV in logs/{exp_dir}/")
    return pd.read_csv(files[-1])


print("Helpers loaded.")

---
## 1  Throughput Ceiling  (`exp01_throughput_sweep.sh`)

**Question:** What is the peak docs/sec achievable on a single GPU, and at which batch size?

A single batch-size sweep with 100 measurement runs per point.

In [ ]:
sweep1 = load_sweep("exp01_throughput_sweep")
sweep1[["batch_size", "docs_per_second_mean", "per_sample_latency_ms_mean",
        "peak_gpu_memory_mb_mean", "decoder_efficiency_mean", "oom_count"]]

In [ ]:
# Figure 1a — Throughput + per-sample latency vs batch size
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: docs/sec
ax = axes[0]
ax.plot(sweep1["batch_size"], sweep1["docs_per_second_mean"],
        marker="o", linewidth=2, label="docs/sec")
best_idx = sweep1["docs_per_second_mean"].idxmax()
best_bs  = sweep1.loc[best_idx, "batch_size"]
best_thr = sweep1.loc[best_idx, "docs_per_second_mean"]
ax.axvline(best_bs, color="red", linestyle="--", alpha=0.6,
           label=f"Optimal B={best_bs}")
ax.annotate(f"{best_thr:.1f} doc/s",
            xy=(best_bs, best_thr), xytext=(best_bs + 0.5, best_thr * 0.9),
            arrowprops=dict(arrowstyle="->", color="red"), color="red")
ax.set_xlabel("Batch size")
ax.set_ylabel("Throughput (docs / sec)")
ax.set_title("GPU Throughput vs Batch Size")
ax.set_xscale("log", base=2)
ax.xaxis.set_major_formatter(mtick.ScalarFormatter())
ax.legend()

# Right: per-sample latency
ax2 = axes[1]
ax2.plot(sweep1["batch_size"], sweep1["per_sample_latency_ms_mean"],
         marker="s", color="orange", linewidth=2, label="mean")
ax2.fill_between(
    sweep1["batch_size"],
    sweep1["per_sample_latency_ms_mean"] - sweep1.get("per_sample_latency_ms_std", 0),
    sweep1["per_sample_latency_ms_mean"] + sweep1.get("per_sample_latency_ms_std", 0),
    alpha=0.2, color="orange", label="±1 std"
)
ax2.axvline(best_bs, color="red", linestyle="--", alpha=0.6)
ax2.set_xlabel("Batch size")
ax2.set_ylabel("Per-sample latency (ms)")
ax2.set_title("Per-Sample Latency vs Batch Size")
ax2.set_xscale("log", base=2)
ax2.xaxis.set_major_formatter(mtick.ScalarFormatter())
ax2.legend()

fig.suptitle("Experiment 1 — Throughput Ceiling", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "exp01_throughput.png", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 1b — Memory and decoder efficiency vs batch size
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.bar(sweep1["batch_size"].astype(str), sweep1["peak_gpu_memory_mb_mean"],
       color=sns.color_palette("husl", len(sweep1)))
ax.set_xlabel("Batch size")
ax.set_ylabel("Peak GPU memory (MB)")
ax.set_title("Peak GPU Memory vs Batch Size")
for i, row in sweep1.iterrows():
    ax.text(i, row["peak_gpu_memory_mb_mean"] + 10,
            f"{row['peak_gpu_memory_mb_mean']:.0f}",
            ha="center", va="bottom", fontsize=9)

ax2 = axes[1]
ax2.plot(sweep1["batch_size"], sweep1["decoder_efficiency_mean"],
         marker="^", color="green", linewidth=2)
ax2.axhline(1.0, color="gray", linestyle=":", label="Perfect efficiency")
ax2.set_xlabel("Batch size")
ax2.set_ylabel("Decoder efficiency  (actual / max tokens)")
ax2.set_title("Decoder Efficiency vs Batch Size")
ax2.set_xscale("log", base=2)
ax2.xaxis.set_major_formatter(mtick.ScalarFormatter())
ax2.set_ylim(0, 1.1)
ax2.legend()

fig.suptitle("Experiment 1 — Memory & Efficiency", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "exp01_memory_efficiency.png", bbox_inches="tight")
plt.show()

---
## 2  Measurement Stability  (`exp02_stability.sh`)

**Question:** Are 10 warmup runs enough? Is our variance acceptable for reliable p95/p99 estimates?

500 runs at B=1 with zero warmup vs. 20 warmup iterations.

In [ ]:
no_wu   = load_single_run_csv("exp02_stability/no_warmup")
with_wu = load_single_run_csv("exp02_stability/with_warmup")

no_wu["variant"]   = "0 warmup"
with_wu["variant"] = "20 warmup"

print(f"no_warmup  rows: {len(no_wu)}  |  with_warmup rows: {len(with_wu)}")

In [ ]:
# Figure 2a — Latency over run index
fig, ax = plt.subplots(figsize=(13, 5))

for df, color, label in [
    (no_wu,   "steelblue", "0 warmup"),
    (with_wu, "coral",     "20 warmup"),
]:
    ax.plot(df["run_index"], df["end_to_end_ms"],
            alpha=0.4, linewidth=0.7, color=color)
    roll = df["end_to_end_ms"].rolling(20, min_periods=1).mean()
    ax.plot(df["run_index"], roll,
            linewidth=2.5, color=color, label=f"{label} (rolling mean 20)")

ax.set_xlabel("Run index")
ax.set_ylabel("End-to-end latency (ms)")
ax.set_title("Latency Over Run Index — Warmup Effect (B=1, 500 runs)")
ax.legend()
plt.tight_layout()
fig.savefig(FIGS / "exp02_latency_trace.png", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2b — Rolling std convergence + latency distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: rolling std convergence for both variants
ax = axes[0]
for df, color, label in [
    (no_wu,   "steelblue", "0 warmup"),
    (with_wu, "coral",     "20 warmup"),
]:
    for window, ls in [(10, ":"), (50, "--"), (100, "-")]:
        rs = df["end_to_end_ms"].rolling(window).std()
        ax.plot(df["run_index"], rs,
                linestyle=ls, color=color, linewidth=1.2, alpha=0.8,
                label=f"{label} w={window}")
ax.set_xlabel("Run index")
ax.set_ylabel("Rolling std of latency (ms)")
ax.set_title("Rolling Std Convergence")
ax.legend(fontsize=8, ncol=2)

# Right: latency histograms
ax2 = axes[1]
for df, color, label in [
    (no_wu,   "steelblue", "0 warmup"),
    (with_wu, "coral",     "20 warmup"),
]:
    ax2.hist(df["end_to_end_ms"], bins=50, alpha=0.55,
             color=color, label=label, density=True, edgecolor="white")
    sns.kdeplot(df["end_to_end_ms"], ax=ax2, color=color, linewidth=2)

ax2.set_xlabel("End-to-end latency (ms)")
ax2.set_ylabel("Density")
ax2.set_title("Latency Distribution")
ax2.legend()

fig.suptitle("Experiment 2 — Measurement Stability", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "exp02_stability.png", bbox_inches="tight")
plt.show()

# Print CV and key percentiles
print(f"\n{'Variant':<15} {'mean':>8} {'std':>8} {'CV%':>7} {'p95':>8} {'p99':>8}")
for df, label in [(no_wu, "0 warmup"), (with_wu, "20 warmup")]:
    s = df["end_to_end_ms"]
    print(f"{label:<15} {s.mean():>8.1f} {s.std():>8.1f} "
          f"{100*s.std()/s.mean():>6.1f}% "
          f"{s.quantile(0.95):>8.1f} {s.quantile(0.99):>8.1f}")

---
## 3  Phase Breakdown  (`exp03_phase_breakdown.sh`)

**Question:** Where does inference time go, and which phase limits throughput scaling?

300 runs per batch size at B=1..32.

In [ ]:
runs3 = load_runs("exp03_phase_breakdown")

phases = ["preprocess_ms", "encoder_ms", "decoder_ms", "postprocess_ms"]
phase_labels = ["Preprocess", "Encoder", "Decoder", "Postprocess"]

phase_means = (
    runs3.groupby("batch_size")[phases]
    .mean()
    .reset_index()
    .sort_values("batch_size")
)
phase_means

In [ ]:
# Figure 3a — Stacked bar: mean phase times per batch size
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

palette = sns.color_palette("Set2", 4)
batch_labels = phase_means["batch_size"].astype(str).tolist()
x = np.arange(len(batch_labels))

# Absolute stacked bar
ax = axes[0]
bottom = np.zeros(len(phase_means))
for col, label, color in zip(phases, phase_labels, palette):
    vals = phase_means[col].values
    ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white")
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(batch_labels)
ax.set_xlabel("Batch size")
ax.set_ylabel("Mean latency (ms)")
ax.set_title("Phase Time Budget (absolute)")
ax.legend()

# Normalised stacked bar (fractions)
ax2 = axes[1]
totals = phase_means[phases].sum(axis=1).values
bottom = np.zeros(len(phase_means))
for col, label, color in zip(phases, phase_labels, palette):
    fracs = phase_means[col].values / totals
    bars = ax2.bar(x, fracs, bottom=bottom, label=label, color=color, edgecolor="white")
    for bar, frac in zip(bars, fracs):
        if frac > 0.05:
            ax2.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_y() + bar.get_height() / 2,
                     f"{frac:.0%}", ha="center", va="center",
                     fontsize=8, color="white", fontweight="bold")
    bottom += fracs
ax2.set_xticks(x)
ax2.set_xticklabels(batch_labels)
ax2.set_xlabel("Batch size")
ax2.set_ylabel("Fraction of total latency")
ax2.set_title("Phase Fraction (normalised)")
ax2.legend()

fig.suptitle("Experiment 3 — Phase Breakdown", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "exp03_phase_stacked.png", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 3b — Encoder vs Decoder scaling curves
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(phase_means["batch_size"], phase_means["encoder_ms"],
        marker="o", linewidth=2, label="Encoder", color="steelblue")
ax.plot(phase_means["batch_size"], phase_means["decoder_ms"],
        marker="s", linewidth=2, label="Decoder", color="coral")
ax.plot(phase_means["batch_size"], phase_means["preprocess_ms"],
        marker="^", linewidth=1.5, linestyle="--", label="Preprocess", color="green")

ax.set_xlabel("Batch size")
ax.set_ylabel("Mean phase time (ms)")
ax.set_title("Phase Time Scaling with Batch Size")
ax.set_xscale("log", base=2)
ax.xaxis.set_major_formatter(mtick.ScalarFormatter())
ax.legend()

# Annotate encoder/decoder ratio at max batch size
last = phase_means.iloc[-1]
ratio = last["decoder_ms"] / last["encoder_ms"]
ax.annotate(f"Dec/Enc ratio\n= {ratio:.1f}×",
            xy=(last["batch_size"], last["decoder_ms"]),
            xytext=(last["batch_size"] * 0.6, last["decoder_ms"] * 1.1),
            fontsize=9, color="coral")

plt.tight_layout()
fig.savefig(FIGS / "exp03_phase_scaling.png", bbox_inches="tight")
plt.show()

---
## 4  CPU vs GPU  (`exp04_cpu_baseline.sh`)

**Question:** How much faster is the GPU? Does CPU batching scale similarly?

CPU sweep at B=1,2,4 compared against the equivalent GPU points from exp01.

In [ ]:
sweep_cpu = load_sweep("exp04_cpu_baseline")
sweep_gpu = load_sweep("exp01_throughput_sweep")

# Keep only batch sizes present in both
common_bs = set(sweep_cpu["batch_size"]) & set(sweep_gpu["batch_size"])
cpu = sweep_cpu[sweep_cpu["batch_size"].isin(common_bs)].set_index("batch_size")
gpu = sweep_gpu[sweep_gpu["batch_size"].isin(common_bs)].set_index("batch_size")

speedup = gpu["docs_per_second_mean"] / cpu["docs_per_second_mean"]
print("GPU speedup over CPU:")
print(speedup.to_string())

In [ ]:
# Figure 4 — CPU vs GPU throughput + speedup
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bs_labels = [str(b) for b in sorted(common_bs)]
x = np.arange(len(bs_labels))
w = 0.35

ax = axes[0]
ax.bar(x - w/2, [cpu.loc[b, "docs_per_second_mean"] for b in sorted(common_bs)],
       w, label="CPU", color="steelblue")
ax.bar(x + w/2, [gpu.loc[b, "docs_per_second_mean"] for b in sorted(common_bs)],
       w, label="GPU", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(bs_labels)
ax.set_xlabel("Batch size")
ax.set_ylabel("Throughput (docs/sec)")
ax.set_title("CPU vs GPU Throughput")
ax.legend()

ax2 = axes[1]
sp_vals = [speedup.loc[b] for b in sorted(common_bs)]
bars = ax2.bar(bs_labels, sp_vals, color=sns.color_palette("husl", len(bs_labels)))
for bar, val in zip(bars, sp_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f"{val:.1f}×", ha="center", va="bottom", fontweight="bold")
ax2.set_xlabel("Batch size")
ax2.set_ylabel("GPU speedup (×)")
ax2.set_title("GPU Speedup Over CPU")

fig.suptitle("Experiment 4 — CPU vs GPU", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "exp04_cpu_vs_gpu.png", bbox_inches="tight")
plt.show()

---
## 5  Decoder Efficiency vs Document Diversity  (`exp05_decoder_efficiency.sh`)

**Question:** How much throughput do we lose by batching diverse documents with variable output lengths?

Uniform pool (same image repeated → no padding waste) vs. full 100-image diverse pool.

In [ ]:
sweep_uniform = load_sweep("exp05_decoder_efficiency/uniform")
sweep_diverse = load_sweep("exp05_decoder_efficiency/diverse")

# Token distribution: load per-run CSV for diverse pool at B=1
# (each run = 1 image, so we see per-image token count)
try:
    runs5_div = load_runs("exp05_decoder_efficiency/diverse", bs=1)
    has_token_dist = True
except FileNotFoundError:
    has_token_dist = False

print("Uniform sweep:")
print(sweep_uniform[["batch_size", "decoder_efficiency_mean", "tokens_per_second_mean"]].to_string())
print("\nDiverse sweep:")
print(sweep_diverse[["batch_size", "decoder_efficiency_mean", "tokens_per_second_mean"]].to_string())

In [ ]:
# Figure 5a — Decoder efficiency comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(sweep_uniform["batch_size"], sweep_uniform["decoder_efficiency_mean"],
        marker="o", linewidth=2, label="Uniform pool", color="steelblue")
ax.plot(sweep_diverse["batch_size"], sweep_diverse["decoder_efficiency_mean"],
        marker="s", linewidth=2, label="Diverse pool", color="coral")
ax.axhline(1.0, color="gray", linestyle=":", label="Perfect efficiency")
ax.set_xlabel("Batch size")
ax.set_ylabel("Decoder efficiency")
ax.set_title("Decoder Efficiency: Uniform vs Diverse")
ax.set_xscale("log", base=2)
ax.xaxis.set_major_formatter(mtick.ScalarFormatter())
ax.set_ylim(0, 1.1)
ax.legend()

ax2 = axes[1]
ax2.plot(sweep_uniform["batch_size"], sweep_uniform["tokens_per_second_mean"],
         marker="o", linewidth=2, label="Uniform pool", color="steelblue")
ax2.plot(sweep_diverse["batch_size"], sweep_diverse["tokens_per_second_mean"],
         marker="s", linewidth=2, label="Diverse pool", color="coral")
ax2.set_xlabel("Batch size")
ax2.set_ylabel("Tokens / sec")
ax2.set_title("Token Throughput: Uniform vs Diverse")
ax2.set_xscale("log", base=2)
ax2.xaxis.set_major_formatter(mtick.ScalarFormatter())
ax2.legend()

fig.suptitle("Experiment 5 — Decoder Efficiency", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "exp05_decoder_efficiency.png", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 5b — Token count distribution across images (B=1 diverse run)
if has_token_dist:
    fig, ax = plt.subplots(figsize=(9, 5))
    tokens = runs5_div["num_generated_tokens"]
    ax.hist(tokens, bins=30, edgecolor="white", color="coral", density=True, alpha=0.8)
    sns.kdeplot(tokens, ax=ax, color="darkred", linewidth=2)
    ax.axvline(tokens.mean(), color="black", linestyle="--",
               label=f"Mean = {tokens.mean():.0f} tokens")
    ax.axvline(tokens.quantile(0.95), color="gray", linestyle=":",
               label=f"p95 = {tokens.quantile(0.95):.0f} tokens")
    ax.set_xlabel("Generated tokens per document")
    ax.set_ylabel("Density")
    ax.set_title("Output Token Count Distribution — cord-v2 test set (B=1)")
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGS / "exp05_token_distribution.png", bbox_inches="tight")
    plt.show()

    print(f"\nToken count stats:  mean={tokens.mean():.0f}  std={tokens.std():.0f}  "
          f"min={tokens.min():.0f}  max={tokens.max():.0f}  "
          f"CV={100*tokens.std()/tokens.mean():.1f}%")
    print("High CV → significant padding waste at large batch sizes.")
else:
    print("Token distribution data not available (run exp05 first).")

---
## 6  Sustained Throughput  (`exp06_sustained_throughput.sh`)

**Question:** Does performance degrade over 1 000 continuous iterations?

Detects thermal throttling, memory fragmentation, or driver scheduling drift.

In [ ]:
runs6_b8  = load_runs("exp06_sustained", bs=8)
runs6_b16 = load_runs("exp06_sustained", bs=16)

for df, bs in [(runs6_b8, 8), (runs6_b16, 16)]:
    print(f"B={bs}: {len(df)} runs  |  "
          f"mean {df['docs_per_second'].mean():.2f} doc/s  |  "
          f"std {df['docs_per_second'].std():.3f}  |  "
          f"drift (last100 - first100 mean): "
          f"{df['docs_per_second'].tail(100).mean() - df['docs_per_second'].head(100).mean():.3f}")

In [ ]:
# Figure 6a — Sustained throughput time series
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for ax, df, bs, color in [
    (axes[0], runs6_b8,  8,  "steelblue"),
    (axes[1], runs6_b16, 16, "coral"),
]:
    ax.plot(df["run_index"], df["docs_per_second"],
            alpha=0.3, linewidth=0.5, color=color)
    roll = df["docs_per_second"].rolling(50, min_periods=1).mean()
    ax.plot(df["run_index"], roll,
            linewidth=2, color=color, label=f"B={bs} (rolling mean 50)")
    # Reference line = mean of last 200 runs (stable regime)
    stable_mean = df["docs_per_second"].tail(200).mean()
    ax.axhline(stable_mean, color="black", linestyle="--", linewidth=1,
               label=f"Stable mean = {stable_mean:.2f}")
    ax.set_ylabel("Docs / sec")
    ax.legend(fontsize=9)
    ax.set_title(f"Sustained Throughput — B={bs}")

axes[1].set_xlabel("Run index")
fig.suptitle("Experiment 6 — Sustained Throughput (1 000 runs)", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "exp06_sustained.png", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 6b — Distribution: first 100 runs vs last 100 runs
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, df, bs in [(axes[0], runs6_b8, 8), (axes[1], runs6_b16, 16)]:
    early = df["end_to_end_ms"].head(100)
    late  = df["end_to_end_ms"].tail(100)
    ax.hist(early, bins=30, alpha=0.6, density=True, label="First 100", color="steelblue", edgecolor="white")
    ax.hist(late,  bins=30, alpha=0.6, density=True, label="Last 100",  color="coral",     edgecolor="white")
    ax.set_xlabel("End-to-end latency (ms)")
    ax.set_ylabel("Density")
    ax.set_title(f"Latency Distribution Drift — B={bs}")
    ax.legend()

fig.suptitle("Experiment 6 — Early vs Late Distribution", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "exp06_distribution_drift.png", bbox_inches="tight")
plt.show()

---
## 7  How Long to Process 100 Documents?  (derived from exp01)

Using the mean end-to-end latency per batch from exp01, we compute the
**minimum wall-clock time to process exactly 100 documents** at each batch size:

$$T_{100}(B) = \left\lceil \frac{100}{B} \right\rceil \times \overline{\text{e2e\_ms}}(B)$$

This accounts for the fact that the last batch may be partial (padded to B).

In [ ]:
sweep1_full = load_sweep("exp01_throughput_sweep")

# We need mean end_to_end_ms per batch size.
# The sweep CSV doesn't carry it directly — load it from per-run CSVs.
runs1 = load_runs("exp01_throughput_sweep")
e2e_by_bs = runs1.groupby("batch_size")["end_to_end_ms"].mean()

records = []
for bs, mean_ms in e2e_by_bs.items():
    n_batches = math.ceil(100 / bs)
    total_ms  = n_batches * mean_ms
    records.append({
        "batch_size":  bs,
        "n_batches":   n_batches,
        "mean_e2e_ms": mean_ms,
        "total_sec":   total_ms / 1000,
    })

df100 = pd.DataFrame(records).sort_values("batch_size").reset_index(drop=True)
best100 = df100.loc[df100["total_sec"].idxmin()]

print(df100.to_string(index=False))
print(f"\n→ Fastest strategy: batch_size={int(best100['batch_size'])}  "
      f"total={best100['total_sec']:.1f}s  "
      f"({int(best100['n_batches'])} batches)")

In [ ]:
# Figure 7 — Total time to process 100 documents
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

palette = sns.color_palette("husl", len(df100))
best_bs = int(best100["batch_size"])
bar_colors = ["red" if bs == best_bs else c
              for bs, c in zip(df100["batch_size"], palette)]

ax = axes[0]
bars = ax.bar(df100["batch_size"].astype(str), df100["total_sec"], color=bar_colors)
ax.set_xlabel("Batch size")
ax.set_ylabel("Estimated total time (sec)")
ax.set_title("Time to Process 100 Documents")
for bar, val in zip(bars, df100["total_sec"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}s", ha="center", va="bottom", fontsize=9)
ax.text(0.5, 0.95, f"Fastest: B={best_bs} ({best100['total_sec']:.1f}s)",
        transform=ax.transAxes, ha="center", va="top",
        fontsize=11, fontweight="bold", color="red")

# Normalised relative to B=1
ax2 = axes[1]
t_b1 = df100.loc[df100["batch_size"] == 1, "total_sec"].values
if len(t_b1):
    norm = df100["total_sec"] / t_b1[0]
    ax2.bar(df100["batch_size"].astype(str), norm, color=bar_colors)
    ax2.axhline(1.0, color="gray", linestyle=":", label="B=1 baseline")
    ax2.set_xlabel("Batch size")
    ax2.set_ylabel("Relative time  (B=1 = 1.0)")
    ax2.set_title("Speedup Relative to B=1")
    ax2.legend()

fig.suptitle("100-Document Processing — Fastest Strategy", fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "exp07_100docs.png", bbox_inches="tight")
plt.show()

---
## 8  Summary Dashboard

Key metrics from all experiments in a single view.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── Panel 1: Throughput vs batch size ─────────────────────────────────────────
ax = axes[0, 0]
ax.plot(sweep1["batch_size"], sweep1["docs_per_second_mean"],
        marker="o", linewidth=2, color="steelblue")
ax.set_xscale("log", base=2)
ax.xaxis.set_major_formatter(mtick.ScalarFormatter())
ax.set_xlabel("Batch size")
ax.set_ylabel("Docs / sec")
ax.set_title("1 · Throughput Ceiling")

# ── Panel 2: Latency distribution (no wu vs with wu) ──────────────────────────
ax = axes[0, 1]
try:
    for df, color, label in [
        (no_wu,   "steelblue", "0 warmup"),
        (with_wu, "coral",     "20 warmup"),
    ]:
        sns.kdeplot(df["end_to_end_ms"], ax=ax, color=color,
                    linewidth=2, label=label)
    ax.set_xlabel("Latency (ms)")
    ax.set_ylabel("Density")
    ax.set_title("2 · Stability: Latency Distribution")
    ax.legend(fontsize=9)
except NameError:
    ax.text(0.5, 0.5, "Run exp02", ha="center", va="center", transform=ax.transAxes)

# ── Panel 3: Phase stacked bar ─────────────────────────────────────────────────
ax = axes[0, 2]
try:
    bottom = np.zeros(len(phase_means))
    for col, label, color in zip(phases, phase_labels, palette):
        ax.bar(phase_means["batch_size"].astype(str),
               phase_means[col], bottom=bottom, label=label, color=color, edgecolor="white")
        bottom += phase_means[col].values
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Latency (ms)")
    ax.set_title("3 · Phase Breakdown")
    ax.legend(fontsize=8)
except NameError:
    ax.text(0.5, 0.5, "Run exp03", ha="center", va="center", transform=ax.transAxes)

# ── Panel 4: Decoder efficiency uniform vs diverse ────────────────────────────
ax = axes[1, 0]
try:
    ax.plot(sweep_uniform["batch_size"], sweep_uniform["decoder_efficiency_mean"],
            marker="o", linewidth=2, label="Uniform", color="steelblue")
    ax.plot(sweep_diverse["batch_size"], sweep_diverse["decoder_efficiency_mean"],
            marker="s", linewidth=2, label="Diverse", color="coral")
    ax.axhline(1.0, color="gray", linestyle=":")
    ax.set_xscale("log", base=2)
    ax.xaxis.set_major_formatter(mtick.ScalarFormatter())
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Decoder efficiency")
    ax.set_title("5 · Decoder Efficiency")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.1)
except NameError:
    ax.text(0.5, 0.5, "Run exp05", ha="center", va="center", transform=ax.transAxes)

# ── Panel 5: Sustained throughput rolling mean ────────────────────────────────
ax = axes[1, 1]
try:
    for df, bs, color in [(runs6_b8, 8, "steelblue"), (runs6_b16, 16, "coral")]:
        roll = df["docs_per_second"].rolling(50, min_periods=1).mean()
        ax.plot(df["run_index"], roll, linewidth=1.5, color=color, label=f"B={bs}")
    ax.set_xlabel("Run index")
    ax.set_ylabel("Docs / sec (rolling 50)")
    ax.set_title("6 · Sustained Throughput")
    ax.legend(fontsize=9)
except NameError:
    ax.text(0.5, 0.5, "Run exp06", ha="center", va="center", transform=ax.transAxes)

# ── Panel 6: 100-doc total time ───────────────────────────────────────────────
ax = axes[1, 2]
try:
    ax.bar(df100["batch_size"].astype(str), df100["total_sec"], color=bar_colors)
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Total time (sec)")
    ax.set_title("7 · Time for 100 Documents")
except NameError:
    ax.text(0.5, 0.5, "Run exp01", ha="center", va="center", transform=ax.transAxes)

fig.suptitle("Donut Inference Benchmark — Summary Dashboard",
             fontsize=16, fontweight="bold")
plt.tight_layout()
fig.savefig(FIGS / "summary_dashboard.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Dashboard saved to {FIGS / 'summary_dashboard.png'}")